# COCO — Microsoft COCO: Common Objects in Context (2014)

_Papers · Datasets, Benchmarks & Surveys_

**A large dataset of everyday scenes where common objects are labeled with per-instance segmentation masks, not just boxes.**

---

This notebook is a practice scaffold for the **COCO — Microsoft COCO: Common Objects in Context (2014)** lesson. We compute the IoU overlap score COCO uses to judge detections, then look at the paper's headline fact about how crowded COCO images are. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Compute IoU, the overlap score COCO scores detections with

COCO judges a detection by **Intersection over Union**: the shared area divided by the combined area (Section 7). A detection counts as correct only when its IoU clears a threshold.

We code the formula and run the lesson's worked example — predicted area 100, true area 120, overlap 80 — then test it against the lenient 0.50 and the strict 0.75 thresholds to see how the same prediction can pass one and fail the other.

In [ ]:
import numpy as np

# IoU (Intersection over Union): the overlap score COCO uses to judge a
# detection (Section 7). IoU = area(overlap) / area(union).
# Worked example from the lesson: predicted mask area 100, true mask area 120,
# overlap 80 pixels.
def iou(area_pred, area_true, overlap):
    union = area_pred + area_true - overlap
    return overlap / union

val = iou(100, 120, 80)
print("IoU(pred=100, true=120, overlap=80) = %.3f" % val)
print("  correct at threshold 0.50? ", val >= 0.50)
print("  correct at threshold 0.75? ", val >= 0.75)
# IoU(pred=100, true=120, overlap=80) = 0.571
#   correct at threshold 0.50?  True
#   correct at threshold 0.75?  False

### Step 2 — Read off the paper's "objects per image" comparison

COCO was built to be crowded and realistic. The paper reports (Section 5, Fig. 5c) that a COCO image carries far more object instances than a PASCAL VOC image.

These two numbers are *transcribed* from the paper, not computed by us. We print them and the ratio to make the design contrast concrete.

In [ ]:
# The paper's QUOTED "objects per image" comparison (Section 5). These two
# numbers are transcribed from the paper, NOT computed by us.
objects_per_image = {"COCO": 7.7, "PASCAL VOC": 2.3}   # Section 5, Fig 5c
for name, n in objects_per_image.items():
    print("%-11s avg instances/image = %.1f  (quoted, Section 5)" % (name, n))

ratio = objects_per_image["COCO"] / objects_per_image["PASCAL VOC"]
print("COCO is %.1fx more crowded per image than PASCAL (from quoted numbers)." % ratio)
# COCO        avg instances/image = 7.7  (quoted, Section 5)
# PASCAL VOC  avg instances/image = 2.3  (quoted, Section 5)
# COCO is 3.3x more crowded per image than PASCAL (from quoted numbers).

## Visualize the data & results

_COCO was built to be crowded and realistic. How many object instances does an average COCO image carry, versus an average PASCAL VOC image? (Both numbers are QUOTED from the paper, Section 5 — this is a schematic, not a run of ours.)_

In [ ]:
# The bar chart shows two numbers QUOTED from the paper (Section 5) — it is a
# schematic of a design fact, not a run of ours.
objects_per_image = {"COCO": 7.7, "PASCAL VOC": 2.3}   # Section 5, Figure 5c
for name, n in objects_per_image.items():
    print("%-11s avg instances/image = %.1f  (quoted from the paper, Section 5)" % (name, n))
print("ratio COCO/PASCAL = %.1fx more crowded per image" % (7.7 / 2.3))
# COCO        avg instances/image = 7.7  (quoted from the paper, Section 5)
# PASCAL VOC  avg instances/image = 2.3  (quoted from the paper, Section 5)
# ratio COCO/PASCAL = 3.3x more crowded per image
# Numbers transcribed from the paper; not generated here.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Why boxes are not enough. Two people stand side by side and overlap so heavily that the smallest
            rectangle around person A almost entirely contains person B as well. (a) Why can a single bounding box
            fail to distinguish a detector that found person A from one that found person B? (b) How does a
            per-instance segmentation mask resolve this? (c) Which sentence in the paper motivates masks over boxes?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Picture both people inside nearly the same rectangle. A box only specifies a region, not which pixels are which person. — _A bounding box throws away shape; two different true objects can share almost the same enclosing rectangle._
- A mask names the exact pixels of one instance, so person A's mask and person B's mask differ even when their boxes coincide. — _Pixel-level outlines separate overlapping instances that boxes cannot._
- Cite §1: boxes "limit the accuracy for which detection algorithms may be evaluated," and the paper proposes "fully segmented instances to enable more accurate detector evaluation." — _The paper's stated reason for choosing masks is precisely this evaluation precision._

**Answer:** (a) A box only marks a rectangle; when A and B share almost the same rectangle, scoring against
                 the box cannot tell which person was actually found &mdash; two different correct/incorrect
                 predictions look the same. (b) A per-instance mask labels the exact pixels of one object, so A's
                 mask and B's mask differ even when their boxes overlap, and the benchmark can score each instance
                 separately. (c) §1: bounding boxes "limit the accuracy for which detection algorithms may be
                 evaluated," motivating "fully segmented instances to enable more accurate detector evaluation."

</details>

**Problem 2.** Compute an IoU. A detector predicts a mask of area 90 pixels for a cat. The human-annotated mask
            has area 150 pixels. They overlap on 75 pixels. (a) Compute the IoU. (b) Is this detection counted as
            correct at an IoU threshold of 0.5? At 0.75? (c) What does this tell you about why COCO averages mAP
            over multiple thresholds?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Intersection is the overlap: 75 pixels. — _IoU's numerator is the area shared by prediction and ground truth._
- Union = 90 + 150 − 75 = 165 pixels. — _Union is both areas minus the double-counted overlap._
- IoU = 75 / 165 ≈ 0.4545. Below 0.5, so it fails even the lenient threshold here. — _A detection counts as correct only when IoU clears the threshold._

**Answer:** (a) Intersection = 75, union = 90 + 150 − 75 = 165, so IoU = 75/165 ≈ 0.45. (b) At a 0.5
                 threshold it is not correct (0.45 < 0.5); at 0.75 it is also not correct. (c) Because a
                 single threshold gives a coarse pass/fail, COCO averages mAP over a range of IoU thresholds: a
                 loose prediction can pass a lenient threshold but fails strict ones, so the averaged score rewards
                 detectors that localize tightly, not just roughly &mdash; the precision the per-instance
                 masks make measurable.

</details>

**Problem 3.** Ablation &mdash; collapse the pipeline. Imagine COCO had skipped stages 1 and 2 and asked crowd
            workers to directly segment every object in each image in one pass (no category labeling, no instance
            spotting first). Predict two concrete ways annotation quality or cost would degrade, and tie each to the
            stage you removed.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Remove stage 1 (category labeling): workers no longer have a checklist of which of the 91 categories are present. — _Stage 1 cheaply establishes "what is here" so later stages know what to look for; without it, rare or small categories get overlooked._
- Remove stage 2 (instance spotting): nobody has marked every separate instance before tracing begins. — _Stage 2 guarantees completeness — all instances located — before the expensive tracing; without it, instances in clutter are missed and never segmented._
- Note the cost effect: the expensive tracing now runs blind on full images instead of on confirmed, located instances. — _The pipeline's ordering exists so the costly step does the minimum work on pre-filtered targets._

**Answer:** Removing stage 1 (category labeling) means workers segment without a list of present
                 categories, so small or uncommon objects are more likely missed entirely &mdash; recall drops.
                 Removing stage 2 (instance spotting) means no one has confirmed every instance before
                 tracing, so in crowded, non-iconic scenes some instances are never outlined &mdash; completeness
                 drops &mdash; while the expensive segmentation step now scans entire images blind, raising cost.
                 The three-stage order exists precisely so the cheap stages filter and locate, and the costly
                 tracing runs only on confirmed, spotted instances (§4).

</details>